# Interactive Bloch Sphere State Visualizer

Set the weights of the orthogonal qubit basis states $|0\rangle$ and $|1\rangle$.
The notebook normalizes the state and plots it on the Bloch sphere.

State form used:
\[
|\psi\rangle = \alpha |0\rangle + \beta |1\rangle,
\]
where
\[
\alpha = w_0,\quad \beta = w_1 e^{i\phi}.
\]


In [4]:
# Fast check: only warns if dependencies are missing (no repeated install).
missing = []

try:
    import ipywidgets  # noqa: F401
except ImportError:
    missing.append('ipywidgets')

try:
    import plotly  # noqa: F401
except ImportError:
    missing.append('plotly')

if not missing:
    print('ipywidgets and plotly are available.')
else:
    print('Missing packages:', ', '.join(missing))
    print('Run this once in a separate cell: %pip install ' + ' '.join(missing))

Missing packages: plotly
Run this once in a separate cell: %pip install plotly


In [5]:
pip install plotly

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 6.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 449.4/449.4 kB 7.2 MB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [8]:
import numpy as np
import plotly.graph_objects as go
from ipywidgets import FloatSlider, interactive_output, VBox
from IPython.display import display, clear_output, HTML

SPHERE_RESOLUTION = 48


def bloch_coordinates(alpha: complex, beta: complex):
    """Convert a normalized qubit state alpha|0> + beta|1> to Bloch coordinates."""
    x = 2 * np.real(np.conjugate(alpha) * beta)
    y = 2 * np.imag(np.conjugate(alpha) * beta)
    z = np.abs(alpha) ** 2 - np.abs(beta) ** 2
    return float(x), float(y), float(z)


def bloch_surface_traces():
    """Return Plotly traces for Bloch sphere and coordinate axes."""
    u = np.linspace(0, 2 * np.pi, SPHERE_RESOLUTION)
    v = np.linspace(0, np.pi, SPHERE_RESOLUTION)

    xs = np.outer(np.cos(u), np.sin(v))
    ys = np.outer(np.sin(u), np.sin(v))
    zs = np.outer(np.ones_like(u), np.cos(v))

    sphere = go.Surface(
        x=xs,
        y=ys,
        z=zs,
        opacity=0.2,
        showscale=False,
        colorscale=[[0, '#8ecae6'], [1, '#8ecae6']],
        hoverinfo='skip',
    )

    x_axis = go.Scatter3d(
        x=[-1, 1], y=[0, 0], z=[0, 0],
        mode='lines', line=dict(color='red', width=6),
        showlegend=False, hoverinfo='skip'
    )
    y_axis = go.Scatter3d(
        x=[0, 0], y=[-1, 1], z=[0, 0],
        mode='lines', line=dict(color='green', width=6),
        showlegend=False, hoverinfo='skip'
    )
    z_axis = go.Scatter3d(
        x=[0, 0], y=[0, 0], z=[-1, 1],
        mode='lines', line=dict(color='blue', width=6),
        showlegend=False, hoverinfo='skip'
    )

    labels = go.Scatter3d(
        x=[1.08, 0.0, 0.0, 0.03, 0.03],
        y=[0.0, 1.08, 0.0, 0.03, 0.03],
        z=[0.0, 0.0, 1.08, 1.03, -1.08],
        mode='text',
        text=['X', 'Y', 'Z', '|0>', '|1>'],
        textfont=dict(size=12, color='black'),
        showlegend=False,
        hoverinfo='skip',
    )

    basis_points = go.Scatter3d(
        x=[0, 0], y=[0, 0], z=[1, -1],
        mode='markers',
        marker=dict(size=4, color='black'),
        showlegend=False,
        hoverinfo='skip',
    )

    return [sphere, x_axis, y_axis, z_axis, labels, basis_points]


def visualize_state(w0: float, w1: float, phi: float):
    # Keep output area lightweight and replace prior figure.
    clear_output(wait=True)

    alpha = complex(w0, 0.0)
    beta = w1 * np.exp(1j * phi)

    norm = np.sqrt(np.abs(alpha) ** 2 + np.abs(beta) ** 2)
    if norm == 0:
        alpha, beta = 1.0 + 0j, 0.0 + 0j
        norm = 1.0

    alpha /= norm
    beta /= norm

    x, y, z = bloch_coordinates(alpha, beta)

    state_line = go.Scatter3d(
        x=[0, x], y=[0, y], z=[0, z],
        mode='lines+markers',
        line=dict(color='purple', width=8),
        marker=dict(size=[0, 6], color='purple'),
        name='State vector',
        hovertemplate='x=%{x:.3f}<br>y=%{y:.3f}<br>z=%{z:.3f}<extra></extra>',
    )

    fig = go.Figure(data=bloch_surface_traces() + [state_line])
    fig.update_layout(
        title='Bloch Sphere (Plotly)',
        margin=dict(l=0, r=0, b=0, t=40),
        scene=dict(
            xaxis=dict(range=[-1.1, 1.1], visible=False),
            yaxis=dict(range=[-1.1, 1.1], visible=False),
            zaxis=dict(range=[-1.1, 1.1], visible=False),
            aspectmode='cube',
            camera=dict(eye=dict(x=1.35, y=1.35, z=0.95)),
        ),
        showlegend=False,
    )

    # Render as HTML to avoid nbformat MIME-renderer dependency issues.
    html = fig.to_html(include_plotlyjs='cdn', full_html=False)
    display(HTML(html))

    print('Normalized state:')
    print(f'|psi> = ({alpha.real:+.4f}{alpha.imag:+.4f}j)|0> + ({beta.real:+.4f}{beta.imag:+.4f}j)|1>')
    print()
    print('Measurement probabilities:')
    print(f'P(0) = {np.abs(alpha) ** 2:.4f}')
    print(f'P(1) = {np.abs(beta) ** 2:.4f}')
    print()
    print('Bloch coordinates:')
    print(f'x = {x:.4f}, y = {y:.4f}, z = {z:.4f}')


In [ ]:
# Interactive controls for state weights and relative phase
w0_slider = FloatSlider(
    value=1.0,
    min=0.0,
    max=2.0,
    step=0.01,
    description='w0',
    continuous_update=False,
)
w1_slider = FloatSlider(
    value=1.0,
    min=0.0,
    max=2.0,
    step=0.01,
    description='w1',
    continuous_update=False,
)
phi_slider = FloatSlider(
    value=0.0,
    min=0.0,
    max=2 * np.pi,
    step=0.01,
    description='phi',
    continuous_update=False,
)

out = interactive_output(
    visualize_state,
    {
        'w0': w0_slider,
        'w1': w1_slider,
        'phi': phi_slider,
    },
)

display(VBox([w0_slider, w1_slider, phi_slider]), out)


Output()